### ANNとは
- 人間の脳の神経回路を模倣した機械学習モデル。
- 複数の層（レイヤー）を持ち、各層のニューロンが重みを介して次の層に信号を伝える。
- 入力層・隠れ層・出力層の3種類の層から構成され、隠れ層を複数重ねることで複雑な非線形関係を学習できる。

||内容|
|----|----|
|モデル|入力層 → 隠れ層 → 出力層|
|損失関数|カテゴリカル交差エントロピー|
|学習|逆伝播 + 勾配降下法|
|出力層の勾配|$ \hat{y}​−y $（ソフトマックス + 交差エントロピーの組み合わせ）|
|注意点|特徴量のスケーリングが必要、重みの初期化に注意|


In [1]:
# ライブラリでの実装
# 3クラス分類を実施

import numpy as np
from sklearn.datasets import load_iris
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 0.9667


In [2]:
# scratchでのANN実装

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler


class ANN:
    def __init__(self, hidden_size=16, lr=0.01, n_epochs=1000, random_state=None):
        self.hidden_size = hidden_size
        self.lr = lr
        self.n_epochs = n_epochs
        self.random_state = random_state

    def _relu(self, z):
        return np.maximum(0, z)

    def _relu_deriv(self, z):
        return (z > 0).astype(float)

    def _softmax(self, z):
        z_shifted = z - z.max(axis=1, keepdims=True)
        exp_z = np.exp(z_shifted)
        return exp_z / exp_z.sum(axis=1, keepdims=True)

    def _one_hot(self, y, n_classes):
        oh = np.zeros((len(y), n_classes))
        oh[np.arange(len(y)), y] = 1
        return oh

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        Y = self._one_hot(y, n_classes)

        self.W1 = rng.standard_normal((n_features, self.hidden_size)) * 0.01
        self.b1 = np.zeros((1, self.hidden_size))
        self.W2 = rng.standard_normal((self.hidden_size, n_classes)) * 0.01
        self.b2 = np.zeros((1, n_classes))

        for _ in range(self.n_epochs):
            # 順伝播
            z1 = X @ self.W1 + self.b1
            a1 = self._relu(z1)
            z2 = a1 @ self.W2 + self.b2
            a2 = self._softmax(z2)

            # 逆伝播
            delta2 = (a2 - Y) / n_samples
            dW2 = a1.T @ delta2
            db2 = delta2.sum(axis=0, keepdims=True)

            delta1 = delta2 @ self.W2.T * self._relu_deriv(z1)
            dW1 = X.T @ delta1
            db1 = delta1.sum(axis=0, keepdims=True)

            # パラメータ更新
            self.W2 -= self.lr * dW2
            self.b2 -= self.lr * db2
            self.W1 -= self.lr * dW1
            self.b1 -= self.lr * db1

    def predict(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = self._relu(z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = self._softmax(z2)
        return np.argmax(a2, axis=1)


# 実行
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

model = ANN(hidden_size=16, lr=0.01, n_epochs=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 0.8000


ソフトマックスのオーバーフロー対策

`z - z.max(axis=1, keepdims=True)` で各サンプルの最大値を引いてからexpを計算しています。expはzが大きいと数値がオーバーフローするため、最大値を引いても確率の値が変わらないことを利用して安定化しています。

重みの初期化

`* 0.01` で小さな値に初期化しています。全ての重みをゼロにすると全ニューロンが同じ勾配を持ち（対称性の破れがない）、学習が進まないためです。

逆伝播の `delta2 = (a2 - Y) / n_samples`

ソフトマックスとカテゴリカル交差エントロピーを組み合わせると出力層の勾配が $ \hat{y}−y $になることを利用しています。`/ n_samples` はミニバッチ平均を取るための正規化です。